Q1. You evaluate two models on 30 cross-validation folds. Model A averages 0.883 and
Model B averages 0.871. Using a paired t-test (scipy.stats.ttest_rel), determine whether this
difference is statistically significant at the 0.05 level. What conclusion do you draw?

In [1]:
import numpy as np
from scipy import stats

np.random.seed(42)

# Simulate 30 paired cross-validation folds centered around their respective means
n_folds = 30
model_a_scores = np.random.normal(loc=0.883, scale=0.015, size=n_folds)
# Model B is systematically lower, but paired to the same random variance per fold
model_b_scores = model_a_scores - 0.012 + np.random.normal(loc=0, scale=0.005, size=n_folds)

# Run the paired t-test
t_stat, p_value = stats.ttest_rel(model_a_scores, model_b_scores)

print(f"Model A Mean: {model_a_scores.mean():.3f}")
print(f"Model B Mean: {model_b_scores.mean():.3f}")
print(f"Paired T-test statistic: {t_stat:.4f}")
print(f"P-value: {p_value:.6f}")

alpha = 0.05
if p_value < alpha:
    print("Conclusion: Reject H₀. The performance difference is statistically significant.")
else:
    print("Conclusion: Fail to reject H₀. The performance difference could be a random fluke.")


Model A Mean: 0.880
Model B Mean: 0.868
Paired T-test statistic: 14.8308
P-value: 0.000000
Conclusion: Reject H₀. The performance difference is statistically significant.


Q2. Explain the difference between a Type I and Type II error in the context of deciding
whether to deploy a new ML model to production. Which error would be more costly in a
medical diagnosis application? In a spam filter?

In [ ]:
. Type I vs. Type II Errors in Model DeploymentIn machine learning product engineering, our testing hypotheses are framed as follows:\(H_{0}\) (Null Hypothesis): The new model is not an improvement (or is worse) compared to the current baseline.\(H_{1}\) (Alternative Hypothesis): The new model is a legitimate improvement over the current system.Error TypeStatistical DefinitionReal-World TranslationType I ErrorRejecting \(H_{0}\) when it is actually true.False Alarm: Deploying a flashy new model that is actually completely useless or worse than your current system.Type II ErrorFailing to reject \(H_{0}\) when \(H_{1}\) is true.Missed Opportunity: Killing a great model in testing and sticking with an inferior legacy system because your test lacked evidence.Application-Specific Cost Analysis:Medical Diagnosis Application: A Type II Error is vastly more costly. If a model fails to catch a malignant tumor (a missed detection), a sick patient goes untreated and faces severe health risks or death. A Type I Error (False Alarm) forces a healthy patient to undergo a secondary screening, which causes anxiety and extra costs, but protects human life.Spam Filter Application: A Type I Error is vastly more costly. If a spam filter registers a false alarm, it takes a critical work email or flight confirmation ticket and hides it inside your junk folder, causing severe real-world disruption. A Type II Error simply lets a stray promotional email leak into your main inbox, which is an easily ignored minor annoyance.

Q3. What does 'statistical power' mean? If your test has only 20% power, what does that
imply about your ability to detect real improvements between models?

In [ ]:
Q3. Understanding Statistical PowerStatistical Power is the mathematical probability that your hypothesis test will successfully detect a real, true effect when one actually exists. Formally, it is calculated as \(\text{Power} = 1 - \beta\), where \(\beta \) represents your Type II error rate.What does 20% Power imply?If your test setup has only 20% power, it means your testing framework is statistically blind.Even if your new model is significantly better than the old one, your test only has a 1-in-5 chance of generating a p-value small enough (\(p < 0.05\)) to prove it.There is an 80% chance (\(\beta = 0.80\)) that you will commit a Type II error, miss the signal entirely, and discard your superior model. Low power is typically caused by having a sample size (\(n\)) that is far too small or a dataset riddled with high background noise.

Q4. A dataset has 10,000 features. You run a t-test on each feature and flag those with p <
0.05 as significant. How many false positives do you expect? What correction technique
(Bonferroni, FDR) would you apply, and why?

In [ ]:
Q4. The False Positive Multiplicity ProblemWhen you perform multiple statistical tests simultaneously, your individual error rates compound rapidly.How many false positives do you expect?The individual significance threshold is \(\alpha = 0.05\). This means that for any single neutral feature, there is a 5% chance it will accidentally spit out a significant p-value by raw mathematical luck.\(\text{Expected\ False\ Positives}=\text{Total\ Features}\times \alpha =10,000\times 0.05=\mathbf{500}\)Without correction, your system will flag 500 completely useless features as "highly important milestones."Correction Techniques:Bonferroni Correction: This is the most conservative solution. It divides your target \(\alpha \) by the total number of tests conducted (\(\alpha_{\text{new}} = 0.05 / 10,000 = 0.000005\)). A feature is only saved if its p-value passes this strict barrier.When to apply: Use when false alarms are strictly forbidden, though it severely tanks your statistical power.False Discovery Rate (FDR / Benjamini-Hochberg Procedure): Instead of trying to prevent any false alarms, FDR controls the proportion of flagged features that are false alarms. If you control FDR at 5%, it guarantees that out of all features you ultimately label as significant, only about 5% of them are flukes.When to apply: FDR is much better for large feature sets (like 10,000 features). It preserves your statistical power, allowing you to discover true hidden signals while screening out the vast majority of mathematical noise

Q5. Generate two samples of size n=100 from Normal(0,1) and Normal(0.3, 1) — a small
but real effect. Run Welch's t-test. What p-value do you get? Now increase n to 1000 and
repeat. Explain why the p-value changes even though the true difference stayed the same.

In [2]:
import numpy as np
from scipy import stats

def run_simulation(n_samples):
    np.random.seed(42)
    # Group 1: Standard Normal
    group_1 = np.random.normal(loc=0.0, scale=1.0, size=n_samples)
    # Group 2: Mean shifted up by a small true effect size of 0.3
    group_2 = np.random.normal(loc=0.3, scale=1.0, size=n_samples)
    
    # Run Welch's T-Test (equal_var=False)
    t_stat, p_val = stats.ttest_ind(group_1, group_2, equal_var=False)
    return t_stat, p_val

# Scenario A: n = 100
t_100, p_100 = run_simulation(100)
print(f"With n = 100:  t-statistic = {t_100:.4f}, p-value = {p_100:.6f}")

# Scenario B: n = 1000
t_1000, p_1000 = run_simulation(1000)
print(f"With n = 1000: t-statistic = {t_1000:.4f}, p-value = {p_1000:.10f}")



Q5. Sample Size and P-Value Dynamics (Welch's T-Test)Let's write the code to execute this simulation at sample sizes of \(n=100\) and \(n=1000\), using Welch's t-test (equal_var=False), which does not assume the two groups share a matching variance.pythonimport numpy as np
Use code with caution.Why does the p-value drop dramatically when \(n\) increases?The true mathematical difference between the two populations stayed perfectly constant at exactly 0.3. However, the p-value drops because p-values are fundamentally driven by sample size.Standard Error Shrinkage: Remember that the Standard Error formula is \(\text{SE} = \sigma / \sqrt{n}\). When you multiply your sample size by 10, your standard error shrinks down. The curves representing your sample means become much narrower and sharper.Signal-to-Noise Ratio: With \(n=100\), a gap of 0.3 is relatively small compared to the wide, overlapping noise of the individual data entries. The test is slightly uncertain if the shift is a fluke.Infinite Data Convergence: With \(n=1000\), your certainty sky-rockets. The random noise cancels itself out across the massive pool of values. The test can see with near-absolute mathematical certainty that the 0.3 shift is real, causing the p-value to plummet towards zero.


With n = 100:  t-statistic = -3.2360, p-value = 0.001421
With n = 1000: t-statistic = -7.9523, p-value = 0.0000000000
